# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()

            if attempt < tries - 1:
                time.sleep(2 ** attempt)

        except requests.exceptions.Timeout:
            if attempt < tries - 1:
                time.sleep(2 ** attempt)
            else:
                raise

    raise RuntimeError("Weather API failed after all retries")

In [2]:
df = get_weather('2026-09-01', '2026-09-07')
print(df.head())

         time  temperature_2m_max  precipitation_sum
0  2026-09-01                34.9                0.1
1  2026-09-02                36.0                0.7
2  2026-09-03                33.0                0.7
3  2026-09-04                35.7                2.4
4  2026-09-05                29.4                2.5


### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
used_fallback = False

try:
    weather = get_weather('2026-09-05', '2026-10-03')
    weekends = weather[
        weather['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()
except (requests.exceptions.RequestException, RuntimeError):
    used_fallback = True
    weekends = None
    print("API unavailable. Using labelled fallback data in Part 3.")

In [4]:
assert len(weekends) == 2
assert set(weekends['time']) == {'2026-09-05', '2026-10-03'}
print("Part 2 checks passed!")

Part 2 checks passed!


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [5]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

used_fallback = False

try:
    weather = get_weather('2026-09-05', '2026-10-03')
    weekends = weather[
        weather['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()
except (requests.exceptions.RequestException, RuntimeError):
    used_fallback = True
    weekends = None
    print("API unavailable. Using labelled fallback data in Part 3.")

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weekends.copy()
    wx['source'] = 'open-meteo'

print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


In [6]:
assert len(wx) == 2
assert 'source' in wx.columns
assert set(wx['source']) == {'open-meteo'}
print("Part 3 checks passed!")

Part 3 checks passed!


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [7]:

wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)
print(wx.dtypes)


         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo
date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [8]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'

print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [9]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date']:%B %d, %Y} was wetter than "
      f"{drier['date']:%B %d, %Y} by {difference:.1f} mm of rainfall.")


October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of rainfall.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [10]:

from datetime import datetime, timezone

print("Endpoint:", BASE)
print("Coordinates: 38.03, -78.51 (Charlottesville, VA)")
print("Date range: 2026-09-05 to 2026-10-03")
print("Retrieved at:", datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'))
print("Data source:", "fallback sample" if used_fallback else "open-meteo")


Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51 (Charlottesville, VA)
Date range: 2026-09-05 to 2026-10-03
Retrieved at: 2026-10-11 22:00 UTC
Data source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather` returned an empty DataFrame instead of raising an error, the merge with 200 days of sales would still run, but the weather columns would contain missing values because there would be no matching weather data. When we chart rainfall against revenue, the chart would either appear empty or exclude all the rows with missing rainfall values, depending on how it was created. The problem is that the code might run without an obvious error, making it seem like the analysis was successful. Nobody might catch it because the sales data would still have 200 rows, and without checking for missing values or validating the merge, we wouldn't realize that the weather data was never retrieved. This is why raising an error is important: it prevents us from drawing conclusions from incomplete or misleading data.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [11]:

orlando = get_weather(
    '2026-09-05', '2026-10-03',
    lat=28.54, lon=-81.38
)

orlando = orlando[
    orlando['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

both_cities = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

both_cities = both_cities[
    ['date', 'city', 'temp_max', 'precip_mm', 'source']
]

print(both_cities)


        date             city  temp_max  precip_mm      source
0 2026-09-05  Charlottesville      29.4        2.5  open-meteo
1 2026-10-03  Charlottesville      22.0       14.7  open-meteo
2 2026-09-05          Orlando      32.1        3.9  open-meteo
3 2026-10-03          Orlando      30.4       14.5  open-meteo
